# LOSO completo en Colab T4 — STFT y CWT

Notebook autocontenido para correr el LOSO **full** (epochs=50, batch=128, AMP) en una GPU T4 de Colab.

**Pre-requisito**: subir el directorio `data/derivatives/modspec_stft_200/` y `data/derivatives/modspec_cwt_200/` a tu Google Drive (~11 GB total). El preproceso y modspec ya están hechos localmente.

Estimado: **~20 min STFT + 20 min CWT = 40 min total** en T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar deps

In [ ]:
!git clone https://github.com/spalaciobe/tps-alzheimer-modspec.git
%cd tps-alzheimer-modspec
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Montar Google Drive y enlazar `data/derivatives/`

Ajusta `DRIVE_DERIV_PATH` a la ruta donde subiste los HDF5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DERIV_PATH = '/content/drive/MyDrive/tps-alzheimer/derivatives'  # ← AJUSTAR

import os
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/derivatives'):
    os.symlink(DRIVE_DERIV_PATH, 'data/derivatives')

!ls -la data/derivatives/

## 4. Copiar HDF5 al SSD efímero (OBLIGATORIO — Drive Mount es muy lento)

Los modulation spectrums son ~11 GB total. Leerlos directamente desde Drive Mount añade segundos de latencia por archivo, lo que hace que cargar el SubjectBank (5-6 GB) tome **varios minutos en lugar de ~30s**. Copiar primero a `/content/` (SSD efímero) es esencial.

In [ ]:
import os, shutil, time

# Copia los modspecs de Drive al SSD efímero de Colab.
# Tarda ~3-5 min (una sola vez), pero acelera el LOSO 10×.
LOCAL_DERIV = '/content/data_local/derivatives'
os.makedirs(LOCAL_DERIV, exist_ok=True)

for d in ('modspec_stft_200', 'modspec_cwt_200'):
    src = f'{DRIVE_DERIV_PATH}/{d}'
    dst = f'{LOCAL_DERIV}/{d}'
    if os.path.exists(dst) and len(os.listdir(dst)) >= 65:
        print(f'{d}: ya copiado ({len(os.listdir(dst))} archivos)', flush=True)
        continue
    if not os.path.exists(src):
        print(f'{d}: NO encontrado en Drive — skip', flush=True)
        continue
    print(f'Copiando {d} a SSD efímero...', flush=True)
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'  {len(os.listdir(dst))} archivos en {time.time()-t0:.0f}s', flush=True)

# Reapunta data/derivatives al SSD efímero
if os.path.islink('data/derivatives'):
    os.unlink('data/derivatives')
elif os.path.exists('data/derivatives'):
    shutil.rmtree('data/derivatives')
os.makedirs('data', exist_ok=True)
os.symlink(LOCAL_DERIV, 'data/derivatives')
print(); print('data/derivatives ahora apunta a SSD efímero')
!ls -la data/derivatives/

## 5. Smoke test rápido (2 folds, ~3-5 min en T4)

Antes del full, verifica que un fold corre OK. Debe aparecer:
- `SubjectBank cargado en ~30s — X.shape=(...)`
- `[fold 00/65] start test=sub-001`
- `[fold 00/65] done train=~40s pred=... true=...`

Si pasa 2 min sin ningún output, hay algo mal — revisa GPU con `!nvidia-smi`.

In [ ]:
!python -u scripts/03_train_loso.py --method stft --fs 200 --seed 0 --max-folds 2

## 6. LOSO STFT full + LOSO CWT full (~40-60 min juntos)

Si el smoke pasó, lanza los completos. **No cambies de pestaña** ni cierres el browser.

Para evitar que Colab desconecte por inactividad, abre F12 → Console y pega:
```javascript
function keepAlive(){
  document.querySelector('colab-connect-button')
    ?.shadowRoot?.querySelector('#connect')?.click();
}
setInterval(keepAlive, 60000);
```

In [ ]:
!python -u scripts/03_train_loso.py --method stft --fs 200 --seed 0
!python -u scripts/03_train_loso.py --method cwt  --fs 200 --seed 0

## 7. Saliency + SVM + Comparación + Figuras

In [ ]:
!python scripts/04_extract_saliency_features.py --method stft --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20
!python scripts/04_extract_saliency_features.py --method cwt  --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20
!python scripts/04_extract_saliency_features.py --method stft --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20 --saliency-method vanilla
!python scripts/04_extract_saliency_features.py --method cwt  --fs 200 --seed 0 --grid-search --max-subjects-per-fold 20 --saliency-method vanilla

In [ ]:
!python scripts/05_run_svm.py --method stft --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80
!python scripts/05_run_svm.py --method cwt  --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80
!python scripts/05_run_svm.py --method stft --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80 --saliency-method vanilla
!python scripts/05_run_svm.py --method cwt  --fs 200 --seed 0 --per-fold-patches --epochs-per-subject 80 --saliency-method vanilla

In [ ]:
!python scripts/06_compare_stft_cwt.py --classifier cnn
!python scripts/06_compare_stft_cwt.py --classifier svm --per-fold
!python scripts/06_compare_stft_cwt.py --classifier svm --per-fold --saliency-method vanilla
!python scripts/07_generate_figures.py --per-fold
!python scripts/07_generate_figures.py --per-fold --saliency-method vanilla

## 8. Copiar resultados de vuelta a Drive

In [ ]:
import shutil
DRIVE_RESULTS = '/content/drive/MyDrive/tps-alzheimer/results-full'
shutil.copytree('results', DRIVE_RESULTS, dirs_exist_ok=True)
shutil.copytree('data/derivatives/saliency', DRIVE_RESULTS + '/saliency', dirs_exist_ok=True)
print('Resultados full guardados en', DRIVE_RESULTS)